# Quantum-AI Drug Discovery — MolGAN Training (Colab)

Train a MolGAN (WGAN-GP) on the ZINC small-molecule dataset and download a
checkpoint that is **byte-compatible with the local Streamlit UI**.

## How to use

1. Set the runtime to GPU: **Runtime › Change runtime type › Hardware accelerator: GPU**.
2. Run cells top-to-bottom.
3. When prompted, upload your `ZINC_250k.csv` (or any CSV with a `smiles` column).
4. Wait for training (5–30 min depending on epochs / dataset size).
5. Download the resulting `molgan.pt` and place it at `checkpoints/molgan.pt` in the local project.
6. Locally: `streamlit run app.py` — the sidebar should say *Checkpoint found.*

The architecture, atom vocab, and config keys here exactly mirror
`src/smiles_graph.py` and `src/models/molgan.py` in the project, so the
checkpoint loads with `MolGAN.load_checkpoint(...)` without any extra glue.

## 0. Environment check

In [ ]:
import torch

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected. Switch runtime to GPU for ~10x speedup.')

## 1. Install dependencies

Colab already ships with PyTorch, NumPy, and pandas. We only need RDKit.

In [ ]:
!pip install -q rdkit

## 2. Upload the ZINC dataset

Pick **one** of the cells below.
- **Option A**: upload directly from your computer (good for one-off runs).
- **Option B**: mount Google Drive (good if the CSV already lives there).

Either way, the file ends up at `/content/data/<name>.csv`.

In [ ]:
# --- Option A: upload from your machine ---
from pathlib import Path

DATA_DIR = Path('/content/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

from google.colab import files
print('Pick your ZINC CSV (e.g. ZINC_250k.csv). The upload may take a minute.')
uploaded = files.upload()
for name, content in uploaded.items():
    target = DATA_DIR / name
    target.write_bytes(content)
    print(f'Saved -> {target}  ({len(content):,} bytes)')

In [ ]:
# --- Option B: mount Drive (uncomment if you prefer this) ---
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/data
# !cp '/content/drive/MyDrive/ZINC_250k.csv' /content/data/

## 3. SMILES ↔ graph tensor encoding

Identical to `src/smiles_graph.py`. **Do not change** the constants — the
local project uses these exact shapes to decode generator output, so any
drift here makes the checkpoint unusable locally.

In [ ]:
import logging
from typing import List, Optional, Tuple

import numpy as np
import torch
from rdkit import Chem, RDLogger

RDLogger.DisableLog('rdApp.*')

MAX_ATOMS = 12
ATOM_MAP = [6, 7, 8, 9, 16, 17]   # C, N, O, F, S, Cl
PAD_INDEX = len(ATOM_MAP)         # 6 == PAD
ATOM_TYPES = len(ATOM_MAP) + 1    # 7
BOND_TYPES = 5                    # NoBond, Single, Double, Triple, Aromatic

BOND_MAP = [
    None,
    Chem.BondType.SINGLE,
    Chem.BondType.DOUBLE,
    Chem.BondType.TRIPLE,
    Chem.BondType.AROMATIC,
]
_BOND_TO_IDX = {b: i for i, b in enumerate(BOND_MAP) if b is not None}


def smiles_to_graph(smiles: str) -> Optional[Tuple[np.ndarray, np.ndarray]]:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumHeavyAtoms() > MAX_ATOMS:
        return None
    nodes = np.zeros((MAX_ATOMS, ATOM_TYPES), dtype=np.float32)
    edges = np.zeros((MAX_ATOMS, MAX_ATOMS, BOND_TYPES), dtype=np.float32)
    for i, atom in enumerate(mol.GetAtoms()):
        z = atom.GetAtomicNum()
        if z not in ATOM_MAP:
            return None
        nodes[i, ATOM_MAP.index(z)] = 1.0
    for j in range(mol.GetNumAtoms(), MAX_ATOMS):
        nodes[j, PAD_INDEX] = 1.0
    edges[:, :, 0] = 1.0
    for bond in mol.GetBonds():
        a, b = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bt = bond.GetBondType()
        if bt not in _BOND_TO_IDX:
            continue
        idx = _BOND_TO_IDX[bt]
        edges[a, b, 0] = 0.0
        edges[b, a, 0] = 0.0
        edges[a, b, idx] = 1.0
        edges[b, a, idx] = 1.0
    return nodes, edges


def encode_smiles_dataset(smiles_list: List[str]):
    nodes, edges, kept = [], [], []
    for smi in smiles_list:
        result = smiles_to_graph(smi)
        if result is None:
            continue
        n, e = result
        nodes.append(n)
        edges.append(e)
        kept.append(smi)
    if not nodes:
        raise ValueError('No SMILES could be encoded.')
    return (
        torch.from_numpy(np.stack(nodes)),
        torch.from_numpy(np.stack(edges)),
        kept,
    )


def _attempt_recovery(rw):
    try:
        mol = rw.GetMol()
        frags = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=False)
    except Exception:
        return None
    best = None
    for frag in frags:
        try:
            Chem.SanitizeMol(frag)
        except Exception:
            continue
        if frag.GetNumAtoms() == 0:
            continue
        if best is None or frag.GetNumAtoms() > best.GetNumAtoms():
            best = frag
    return best


def graph_to_mol(nodes: np.ndarray, edges: np.ndarray):
    node_idx = np.argmax(nodes, axis=-1)
    edge_idx = np.argmax(edges, axis=-1)
    rw = Chem.RWMol()
    rdkit_indices = []
    for v in range(MAX_ATOMS):
        atype = int(node_idx[v])
        if atype == PAD_INDEX:
            rdkit_indices.append(None)
            continue
        atom = Chem.Atom(ATOM_MAP[atype])
        rdkit_indices.append(rw.AddAtom(atom))
    for i in range(MAX_ATOMS):
        if rdkit_indices[i] is None:
            continue
        for j in range(i + 1, MAX_ATOMS):
            if rdkit_indices[j] is None:
                continue
            bidx = int(edge_idx[i, j])
            if bidx == 0 or bidx >= len(BOND_MAP) or BOND_MAP[bidx] is None:
                continue
            rw.AddBond(rdkit_indices[i], rdkit_indices[j], BOND_MAP[bidx])
    mol = rw.GetMol()
    try:
        Chem.SanitizeMol(mol)
    except Exception:
        return _attempt_recovery(rw)
    if mol.GetNumAtoms() == 0:
        return None
    return mol


def graph_to_smiles(nodes: np.ndarray, edges: np.ndarray) -> Optional[str]:
    mol = graph_to_mol(nodes, edges)
    if mol is None or mol.GetNumAtoms() == 0:
        return None
    try:
        frags = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=False)
    except Exception:
        return None
    best = None
    for frag in frags:
        try:
            Chem.SanitizeMol(frag)
        except Exception:
            continue
        if frag.GetNumHeavyAtoms() < 2:
            continue
        if best is None or frag.GetNumHeavyAtoms() > best.GetNumHeavyAtoms():
            best = frag
    if best is None:
        return None
    try:
        smi = Chem.MolToSmiles(best, canonical=True)
    except Exception:
        return None
    if not smi or '.' in smi:
        return None
    return smi


def batch_decode_to_smiles(nodes_batch: torch.Tensor, edges_batch: torch.Tensor):
    nodes_np = nodes_batch.detach().cpu().numpy()
    edges_np = edges_batch.detach().cpu().numpy()
    return [graph_to_smiles(nodes_np[i], edges_np[i]) for i in range(nodes_np.shape[0])]

print('Encoder ready. ATOM_TYPES =', ATOM_TYPES, ' BOND_TYPES =', BOND_TYPES)

## 4. MolGAN model

Same `_Generator`, `_Discriminator`, and `MolGAN` as `src/models/molgan.py`.
The `save_checkpoint` payload format matches `MolGAN.load_checkpoint` locally.

In [ ]:
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Callable, List, Optional, Union

import torch.autograd as autograd
import torch.nn as nn
import torch.optim as optim

def _log(msg: str) -> None:
    """Print so Colab actually shows it (logging.basicConfig is a no-op here)."""
    print(msg, flush=True)


@dataclass
class MolGANConfig:
    z_dim: int = 32
    g_hidden: int = 256
    d_hidden: int = 256
    lr_g: float = 1e-4
    lr_d: float = 1e-4
    betas: tuple = (0.5, 0.9)
    n_critic: int = 5
    gp_lambda: float = 10.0
    batch_size: int = 64


class _Generator(nn.Module):
    def __init__(self, cfg: MolGANConfig):
        super().__init__()
        h = cfg.g_hidden
        self.trunk = nn.Sequential(
            nn.Linear(cfg.z_dim, h),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(h, h * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(h * 2, h * 4),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.edge_head = nn.Linear(h * 4, MAX_ATOMS * MAX_ATOMS * BOND_TYPES)
        self.node_head = nn.Linear(h * 4, MAX_ATOMS * ATOM_TYPES)

    def forward(self, z):
        h = self.trunk(z)
        e_logits = self.edge_head(h).view(-1, MAX_ATOMS, MAX_ATOMS, BOND_TYPES)
        n_logits = self.node_head(h).view(-1, MAX_ATOMS, ATOM_TYPES)
        e_logits = 0.5 * (e_logits + e_logits.transpose(1, 2))
        return torch.softmax(e_logits, dim=-1), torch.softmax(n_logits, dim=-1)


class _Discriminator(nn.Module):
    def __init__(self, cfg: MolGANConfig):
        super().__init__()
        in_dim = (MAX_ATOMS * MAX_ATOMS * BOND_TYPES) + (MAX_ATOMS * ATOM_TYPES)
        h = cfg.d_hidden
        self.net = nn.Sequential(
            nn.Linear(in_dim, h * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            nn.Linear(h * 2, h),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(h, 1),
        )

    def forward(self, edges, nodes):
        x = torch.cat([edges.flatten(1), nodes.flatten(1)], dim=1)
        return self.net(x)


class MolGAN:
    def __init__(self, cfg: Optional[MolGANConfig] = None, device: Optional[str] = None):
        self.cfg = cfg or MolGANConfig()
        self.device = torch.device(device or ('cuda' if torch.cuda.is_available() else 'cpu'))
        self.generator = _Generator(self.cfg).to(self.device)
        self.discriminator = _Discriminator(self.cfg).to(self.device)
        self.opt_g = optim.Adam(self.generator.parameters(), lr=self.cfg.lr_g, betas=self.cfg.betas)
        self.opt_d = optim.Adam(self.discriminator.parameters(), lr=self.cfg.lr_d, betas=self.cfg.betas)

    def train(self, smiles_list, epochs=100, log_every=10):
        nodes_real, edges_real, kept = encode_smiles_dataset(smiles_list)
        if len(kept) == 0:
            _log('No SMILES survived encoding; aborting.')
            return
        nodes_real = nodes_real.to(self.device)
        edges_real = edges_real.to(self.device)
        n_samples = nodes_real.shape[0]
        bs = min(self.cfg.batch_size, n_samples)
        n_batches = max(1, n_samples // bs)
        _log(
            f'Training on {n_samples} molecules | device={self.device.type} '
            f'| epochs={epochs} | batches/epoch={n_batches}'
        )
        for epoch in range(1, epochs + 1):
            perm = torch.randperm(n_samples, device=self.device)
            d_total, g_total = 0.0, 0.0
            for b in range(n_batches):
                idx = perm[b * bs : (b + 1) * bs]
                real_edges = edges_real[idx]
                real_nodes = nodes_real[idx]
                cur_bs = real_edges.size(0)
                for _ in range(self.cfg.n_critic):
                    self.opt_d.zero_grad(set_to_none=True)
                    d_real = self.discriminator(real_edges, real_nodes).mean()
                    z = torch.randn(cur_bs, self.cfg.z_dim, device=self.device)
                    fake_edges, fake_nodes = self.generator(z)
                    d_fake = self.discriminator(fake_edges.detach(), fake_nodes.detach()).mean()
                    gp = self._gradient_penalty(real_edges, real_nodes, fake_edges.detach(), fake_nodes.detach())
                    d_loss = d_fake - d_real + self.cfg.gp_lambda * gp
                    d_loss.backward()
                    self.opt_d.step()
                    d_total += d_loss.item()
                self.opt_g.zero_grad(set_to_none=True)
                z = torch.randn(cur_bs, self.cfg.z_dim, device=self.device)
                fake_edges, fake_nodes = self.generator(z)
                g_loss = -self.discriminator(fake_edges, fake_nodes).mean()
                g_loss.backward()
                self.opt_g.step()
                g_total += g_loss.item()
            if epoch == 1 or epoch % log_every == 0 or epoch == epochs:
                d_avg = d_total / (n_batches * self.cfg.n_critic)
                g_avg = g_total / n_batches
                validity = self._spot_check_validity(64)
                _log(
                    f'Epoch {epoch:3d}/{epochs} | D {d_avg:+0.4f} | '
                    f'G {g_avg:+0.4f} | valid {validity:.1f}%'
                )

    @torch.no_grad()
    def sample_smiles(self, n: int, max_tries: int = 10):
        self.generator.eval()
        out, seen = [], set()
        batch = max(n * 2, 32)
        for attempt in range(1, max_tries + 1):
            if len(out) >= n:
                break
            z = torch.randn(batch, self.cfg.z_dim, device=self.device)
            edges, nodes = self.generator(z)
            for smi in batch_decode_to_smiles(nodes, edges):
                if not smi or smi in seen:
                    continue
                seen.add(smi)
                out.append(smi)
                if len(out) >= n:
                    break
        self.generator.train()
        return out[:n]

    @torch.no_grad()
    def _spot_check_validity(self, batch: int = 64) -> float:
        self.generator.eval()
        z = torch.randn(batch, self.cfg.z_dim, device=self.device)
        edges, nodes = self.generator(z)
        decoded = batch_decode_to_smiles(nodes, edges)
        self.generator.train()
        valid = sum(1 for s in decoded if s)
        return 100.0 * valid / batch

    def _gradient_penalty(self, real_edges, real_nodes, fake_edges, fake_nodes):
        bs = real_edges.size(0)
        alpha = torch.rand(bs, 1, 1, 1, device=self.device)
        interp_edges = alpha * real_edges + (1 - alpha) * fake_edges
        interp_nodes = alpha.squeeze(-1) * real_nodes + (1 - alpha.squeeze(-1)) * fake_nodes
        interp_edges.requires_grad_(True)
        interp_nodes.requires_grad_(True)
        d_interp = self.discriminator(interp_edges, interp_nodes)
        grads = autograd.grad(
            outputs=d_interp,
            inputs=(interp_edges, interp_nodes),
            grad_outputs=torch.ones_like(d_interp),
            create_graph=True,
            retain_graph=True,
            only_inputs=True,
        )
        flat = torch.cat([grads[0].flatten(1), grads[1].flatten(1)], dim=1)
        norm = flat.norm(2, dim=1)
        return ((norm - 1) ** 2).mean()

    def save_checkpoint(self, path):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        torch.save({
            'config': asdict(self.cfg),
            'generator': self.generator.state_dict(),
            'discriminator': self.discriminator.state_dict(),
            'opt_g': self.opt_g.state_dict(),
            'opt_d': self.opt_d.state_dict(),
        }, path)
        print(f'Checkpoint saved -> {path}')

print('MolGAN ready.')

## 5. Load and filter the ZINC dataset

Mirrors `src/data_loader.py`: keep canonical SMILES with ≤ 12 heavy atoms
and atoms drawn only from `{C, N, O, F, S, Cl}`.

In [ ]:
import pandas as pd

DATASET_SIZE = 20000
MAX_HEAVY_ATOMS = 12
_SMILES_COLS = ('smiles', 'SMILES', 'Smiles', 'canonical_smiles', 'smi')

def discover_csv(data_dir: Path = DATA_DIR) -> Optional[Path]:
    for path in sorted(data_dir.glob('*.csv')):
        try:
            head = pd.read_csv(path, nrows=5, sep=None, engine='python')
        except Exception:
            continue
        for col in head.columns:
            if str(col).strip().lower() in {c.lower() for c in _SMILES_COLS}:
                return path
    return None

csv_path = discover_csv()
assert csv_path is not None, 'No CSV with a SMILES column found in /content/data/'
print('Using:', csv_path)

df = pd.read_csv(csv_path, nrows=250_000, sep=None, engine='python')
smiles_col = next(c for c in df.columns if str(c).strip().lower() in {x.lower() for x in _SMILES_COLS})
df = df.rename(columns={smiles_col: 'smiles'})
df['smiles'] = df['smiles'].astype(str).str.replace('\n', '', regex=False).str.strip()
df = df[df['smiles'].apply(lambda x: bool(x) and '"' not in x)]

valid = []
for raw in df['smiles']:
    mol = Chem.MolFromSmiles(raw)
    if mol is None or mol.GetNumHeavyAtoms() > MAX_HEAVY_ATOMS:
        continue
    valid.append(Chem.MolToSmiles(mol, canonical=True))
    if len(valid) >= DATASET_SIZE:
        break

print(f'Filtered corpus size: {len(valid)}')
print('First 5 examples:', valid[:5])

## 6. Train

Tune `EPOCHS` based on how much time you have. Roughly:

| Epochs | Time on T4 GPU | Quality |
|---|---|---|
| 50 | ~2 min | low (smoke test) |
| 200 | ~10 min | decent |
| 500–1000 | ~30–60 min | good |

Keep an eye on the **valid %** column: that's the fraction of generator
samples that decode to a sanitizable molecule. It should climb past ~30–50%
with enough training.

In [ ]:
EPOCHS = 300

molgan = MolGAN()
print('Device:', molgan.device)

molgan.train(valid, epochs=EPOCHS, log_every=10)

## 7. Quick sanity check

In [ ]:
samples = molgan.sample_smiles(20)
print(f'Sampled {len(samples)} unique valid SMILES:')
for s in samples:
    print(' -', s)

## 8. Save and download the checkpoint

The downloaded `molgan.pt` should be placed at `checkpoints/molgan.pt` in
the local project. The local Streamlit UI loads it via
`MolGAN.load_checkpoint('checkpoints/molgan.pt')` automatically.

In [ ]:
CHECKPOINT_PATH = '/content/molgan.pt'
molgan.save_checkpoint(CHECKPOINT_PATH)

from google.colab import files
files.download(CHECKPOINT_PATH)

## 9. (Optional) Save to Google Drive instead

Useful for big training runs you want to keep around.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive', force_remount=False)
# !mkdir -p /content/drive/MyDrive/quantum_dd_checkpoints
# !cp /content/molgan.pt /content/drive/MyDrive/quantum_dd_checkpoints/molgan.pt
# print('Saved to Drive: MyDrive/quantum_dd_checkpoints/molgan.pt')